In [ ]:
import gymnasium as gym
import numpy as np
import subprocess
import os
import wave
from gymnasium.wrappers import RecordVideo
from gymnasium.wrappers import ResizeObservation
from sdlarch_rl import make
import pygame
import cv2
import time

import numpy as np

os.makedirs("videos", exist_ok=True)


video_filename = "videos/video-episode-0.mp4"
audio_filename = "videos/game_audio.wav"

audio_data = b""
arate = None
framerate = None

data = {
    "audio_data": audio_data,
    "arate": arate,
    "framerate": framerate,
}

        
#env = make("GranTurismo3-Ps2")
env = make("VirtuaTennis-DC")
env = ResizeObservation(env, (480, 640))
env = RecordVideo(
    env,
    video_folder="videos/",
    episode_trigger=lambda x: x == 0,
    name_prefix="video"
)


render_mode = "rgb_array"
# render_mode = "human"


obs, info = env.reset()
done = False
count = 0

pygame.init()

SCREEN_WIDTH = 640
SCREEN_HEIGHT = 480
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

data['arate'] = env.unwrapped.em.get_audio_rate()
data['framerate'] = env.unwrapped.em.get_frame_rate()

frame_time = 1.0 / data['framerate']
last_time = time.time()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True
            
    keys = pygame.key.get_pressed()

    env.render()

    action = np.zeros(16, dtype=np.uint8)

    if keys[pygame.K_UP]:
        action[4] = 1
    if keys[pygame.K_DOWN]:
        action[5] = 1
    if keys[pygame.K_LEFT]:
        action[6] = 1
    if keys[pygame.K_RIGHT]:
        action[7] = 1
    if keys[pygame.K_c]:
        action[1] = 1
    if keys[pygame.K_x]:
        action[0] = 1
    if keys[pygame.K_RETURN]:
        action[3] = 1
    if keys[pygame.K_l]:
        action[11] = 1

    # clock.tick(60)

    img, rew, done, _, info = env.step(action)

    sound = env.unwrapped.em.get_audio()
    data['audio_data'] += sound.tobytes()

     # stop
    if keys[pygame.K_BACKSPACE]:
        print("== DONE ==")
        done = True

    # frame rate
    now = time.time()
    sleep_time = frame_time - (now - last_time)
    if sleep_time > 0:
        time.sleep(sleep_time)
    last_time = now

    img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
     
    surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))


    window.blit(surface, (0, 0))
    pygame.display.update()

    count += 1

    if done:
        break

env.close()
pygame.quit()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

# save audio
with wave.open(audio_filename, "wb") as wf:
    wf.setnchannels(2)  # Mono
    wf.setsampwidth(2)  # 16-bit PCM
    wf.setframerate(data['arate'])  # Audio rate
    wf.writeframes(data['audio_data'])

video_prefix = "video.mp4"


ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-r", str(data['framerate']),
    "-i", video_filename,
    "-i", audio_filename,
    # "-vf", "scale=1280:720", # hd resolution
    "-vf", "scale=1280:720,setdar=16:9,setsar=1", # HD
    # "-vf", "scale=854:480,setdar=16:9,setsar=1", # 16x9
    "-c:v", "libx265",
    # "-preset", "veryslow",
     "-crf", "10", # quality 10 ~ 50 (10 is better)
    "-c:a", "aac",
    "-b:a", "128k",
    "-ac","2",
    "-strict", "experimental",
    "-shortest",
    "videos/" + video_prefix
]

subprocess.run(ffmpeg_cmd)

print("✅ Finished vídeos")

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Detected game: virtuatennis-dc
Variable: reicast_region = USA

Variable: reicast_language = English

Variable: reicast_hle_bios = disabled

Variable: reicast_boot_to_bios = disabled

Variable: reicast_enable_dsp = enabled

Variable: reicast_allow_service_buttons = disabled

Variable: reicast_force_freeplay = enabled

Variable: reicast_emulate_bba = disabled

Variable: reicast_upnp = enabled

Variable: reicast_dcnet = disabled

Variable: reicast_internal_resolution = 640x480

Variable: reicast_cable_type = TV (Composite)

Variable: reicast_broadcast = NTSC

Variable: reicast_screen_rotation = horizontal

Variable: reicast_alpha_sorting = per-triangle (normal)

Variable: reicast_oit_abuffer_size = 512MB

Variable: reicast_oit_layers = 32

Variable: reicast_emulate_framebuffer = disabled

Variable: reicast_enable_rttb = disabled

Variable: reicast_mipmapping = enabled

Variable: reicast_fog = enabled

Variable: reicast_volume_modifier_enable = enabled

Variable: reicast_anisotropic_filter

D:\Python311\Lib\site-packages\gymnasium\wrappers\record_video.py:94: UserWarning: WARN: Overwriting existing videos at D:\projects\sdlarch\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
